In [1]:
from __future__ import annotations

import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
from facter.config import Config
from facter.data import DatasetLoader
from facter.models import load_models
from facter.fairness import ConformalFairnessValidator, _group_key
from facter.prompt_engine import FairPromptEngine
from facter.utils import setup_logging, generate_recommendations, evaluate_at_k_from_lists, evaluate_valid_at_k

from facter.catalog_map import CatalogMapper
from facter.metrics_fairness import compute_snsr_snsv, compute_cfr
from facter.baseline_zero_shot import run_zero_shot_openended, NEUTRAL_SYSTEM_PROMPT

/opt/miniconda3/envs/facter/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import argparse

import logging
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 3000)  # display long text dfs

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu")

## LLM: Mistral 7B

In [4]:
logger = setup_logging()
np.random.seed(Config.RANDOM_SEED)

In [5]:
embedder, tokenizer, model = load_models(prefer_public_finetuned_embedder=True)

2026-01-16 20:43:21,812 - INFO - Loading embedder: JJTsao/fine-tuned_movie_retriever-all-mpnet-base-v2
2026-01-16 20:43:21,815 - INFO - Use pytorch device_name: mps
2026-01-16 20:43:21,815 - INFO - Load pretrained SentenceTransformer: JJTsao/fine-tuned_movie_retriever-all-mpnet-base-v2
Invalid model-index. Not loading eval results into CardData.
2026-01-16 20:43:23,888 - WARNING - Invalid model-index. Not loading eval results into CardData.
2026-01-16 20:43:24,505 - INFO - Loading LLM: mistralai/Mistral-7B-Instruct-v0.1
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:21<00:00, 10.53s/it]


In [6]:
results = {}

### Dataset: MovieLens

In [7]:
dataset_name = 'ml-1m'

In [8]:
logger.info(f"\n=== Running {dataset_name.upper()} ===")
loader = DatasetLoader(dataset_name)
df = loader.prepare_prompts().dropna().reset_index(drop=True)

# Stratify by full tuple for stable eval
strata = df[Config.PROTECTED_ATTRIBUTES].astype(str).agg("_".join, axis=1)
df = df[strata.map(strata.value_counts()) >= 2].copy()

2026-01-16 20:44:01,128 - INFO - 
=== Running ML-1M ===
Building sequences (ml-1m): 100%|██████████| 6040/6040 [00:04<00:00, 1385.86it/s]


In [9]:
print(len(df))
df.head()

939809


,prompt,context,gender,age,occupation,target_mid,target_title
0,"User profile (audit only):\n- gender: F\n- age: Under 18\n- occupation: K-12 student\n\nWatch history:\n1. Girl, Interrupted (1999)\n2. Back to the Future (1985)\n3. Titanic (1997)\n4. Cinderella (1950)\n5. Meet Joe Black (1998)\n6. Last Days of Disco, The (1998)\n7. Erin Brockovich (2000)\n8. Christmas Story, A (1983)\n9. To Kill a Mockingbird (1962)\n10. One Flew Over the Cuckoo's Nest (1975)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Girl, Interrupted (1999)\n2. Back to the Future (1985)\n3. Titanic (1997)\n4. Cinderella (1950)\n5. Meet Joe Black (1998)\n6. Last Days of Disco, The (1998)\n7. Erin Brockovich (2000)\n8. Christmas Story, A (1983)\n9. To Kill a Mockingbird (1962)\n10. One Flew Over the Cuckoo's Nest (1975)",F,Under 18,K-12 student,720,Wallace & Gromit: The Best of Aardman Animation (1996)
1,"User profile (audit only):\n- gender: F\n- age: Under 18\n- occupation: K-12 student\n\nWatch history:\n1. Back to the Future (1985)\n2. Titanic (1997)\n3. Cinderella (1950)\n4. Meet Joe Black (1998)\n5. Last Days of Disco, The (1998)\n6. Erin Brockovich (2000)\n7. Christmas Story, A (1983)\n8. To Kill a Mockingbird (1962)\n9. One Flew Over the Cuckoo's Nest (1975)\n10. Wallace & Gromit: The Best of Aardman Animation (1996)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Back to the Future (1985)\n2. Titanic (1997)\n3. Cinderella (1950)\n4. Meet Joe Black (1998)\n5. Last Days of Disco, The (1998)\n6. Erin Brockovich (2000)\n7. Christmas Story, A (1983)\n8. To Kill a Mockingbird (1962)\n9. One Flew Over the Cuckoo's Nest (1975)\n10. Wallace & Gromit: The Best of Aardman Animation (1996)",F,Under 18,K-12 student,260,Star Wars: Episode IV - A New Hope (1977)
2,"User profile (audit only):\n- gender: F\n- age: Under 18\n- occupation: K-12 student\n\nWatch history:\n1. Titanic (1997)\n2. Cinderella (1950)\n3. Meet Joe Black (1998)\n4. Last Days of Disco, The (1998)\n5. Erin Brockovich (2000)\n6. Christmas Story, A (1983)\n7. To Kill a Mockingbird (1962)\n8. One Flew Over the Cuckoo's Nest (1975)\n9. Wallace & Gromit: The Best of Aardman Animation (1996)\n10. Star Wars: Episode IV - A New Hope (1977)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Titanic (1997)\n2. Cinderella (1950)\n3. Meet Joe Black (1998)\n4. Last Days of Disco, The (1998)\n5. Erin Brockovich (2000)\n6. Christmas Story, A (1983)\n7. To Kill a Mockingbird (1962)\n8. One Flew Over the Cuckoo's Nest (1975)\n9. Wallace & Gromit: The Best of Aardman Animation (1996)\n10. Star Wars: Episode IV - A New Hope (1977)",F,Under 18,K-12 student,919,"Wizard of Oz, The (1939)"
3,"User profile (audit only):\n- gender: F\n- age: Under 18\n- occupation: K-12 student\n\nWatch history:\n1. Cinderella (1950)\n2. Meet Joe Black (1998)\n3. Last Days of Disco, The (1998)\n4. Erin Brockovich (2000)\n5. Christmas Story, A (1983)\n6. To Kill a Mockingbird (1962)\n7. One Flew Over the Cuckoo's Nest (1975)\n8. Wallace & Gromit: The Best of Aardman Animation (1996)\n9. Star Wars: Episode IV - A New Hope (1977)\n10. Wizard of Oz, The (1939)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Cinderella (1950)\n2. Meet Joe Black (1998)\n3. Last Days of Disco, The (1998)\n4. Erin Brockovich (2000)\n5. Christmas Story, A (1983)\n6. To Kill a Mockingbird (1962)\n7. One Flew Over the Cuckoo's Nest (1975)\n8. Wallace & Gromit: The Best of Aardman Animation (1996)\n9. Star Wars: Episode IV - A New Hope (1977)\n10. Wizard of Oz, The (1939)",F,Under 18,K-12 student,608,Fargo 

In [10]:
train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    random_state=Config.RANDOM_SEED,
    stratify=df[Config.PROTECTED_ATTRIBUTES].astype(str).agg("_".join, axis=1),
)

In [11]:
print(len(train_df))
train_df.head()

657866


,prompt,context,gender,age,occupation,target_mid,target_title
160383,"User profile (audit only):\n- gender: F\n- age: Under 18\n- occupation: K-12 student\n\nWatch history:\n1. Snow White and the Seven Dwarfs (1937)\n2. Fantasia 2000 (1999)\n3. Cell, The (2000)\n4. Road to El Dorado, The (2000)\n5. Big Momma's House (2000)\n6. Gossip (2000)\n7. Chasing Amy (1997)\n8. Dogma (1999)\n9. What Lies Beneath (2000)\n10. Scream (1996)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Snow White and the Seven Dwarfs (1937)\n2. Fantasia 2000 (1999)\n3. Cell, The (2000)\n4. Road to El Dorado, The (2000)\n5. Big Momma's House (2000)\n6. Gossip (2000)\n7. Chasing Amy (1997)\n8. Dogma (1999)\n9. What Lies Beneath (2000)\n10. Scream (1996)",F,Under 18,K-12 student,920,Gone with the Wind (1939)
181219,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: programmer\n\nWatch history:\n1. Thin Red Line, The (1998)\n2. G.I. Jane (1997)\n3. In the Army Now (1994)\n4. Terminator 2: Judgment Day (1991)\n5. Hunt for Red October, The (1990)\n6. Fugitive, The (1993)\n7. Matrix, The (1999)\n8. Speed (1994)\n9. Jurassic Park (1993)\n10. Total Recall (1990)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Thin Red Line, The (1998)\n2. G.I. Jane (1997)\n3. In the Army Now (1994)\n4. Terminator 2: Judgment Day (1991)\n5. Hunt for Red October, The (1990)\n6. Fugitive, The (1993)\n7. Matrix, The (1999)\n8. Speed (1994)\n9. Jurassic Park (1993)\n10. Total Recall (1990)",M,25-34,programmer,733,"Rock, The (1996)"
35914,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: academic/educator\n\nWatch history:\n1. Honey, I Shrunk the Kids (1989)\n2. Armageddon (1998)\n3. Highlander III: The Sorcerer (1994)\n4. Transformers: The Movie, The (1986)\n5. Conquest of the Planet of the Apes (1972)\n6. Beneath the Planet of the Apes (1970)\n7. Coneheads (1993)\n8. Johnny Mnemonic (1995)\n9. Tank Girl (1995)\n10. Honey, I Blew Up the Kid (1992)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Honey, I Shrunk the Kids (1989)\n2. Armageddon (1998)\n3. Highlander III: The Sorcerer (1994)\n4. Transformers: The Movie, The (1986)\n5. Conquest of the Planet of the Apes (1972)\n6. Beneath the Planet of the Apes (1970)\n7. Coneheads (1993)\n8. Johnny Mnemonic (1995)\n9. Tank Girl (1995)\n10. Honey, I Blew Up the Kid (1992)",F,25-34,academic/educator,2091,Return from Witch Mountain (1978)
674709,"User profile (audit only):\n- gender: F\n- age: 18-24\n- occupation: college/grad student\n\nWatch history:\n1. Tarzan (1999)\n2. 101 Dalmatians (1961)\n3. Mulan (1998)\n4. Prince of Egypt, The (1998)\n5. Beavis and Butt-head Do America (1996)\n6. Pocahontas (1995)\n7. Blade Runner (1982)\n8. Matrix, The (1999)\n9. Twelve Monkeys (1995)\n10. Terminator 2: Judgment Day (1991)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Tarzan (1999)\n2. 101 Dalmatians (1961)\n3. Mulan (1998)\n4. Prince of Egypt, The (1998)\n5. Beavis and Butt-head Do America (1996)\n6. Pocahontas (1995)\n7. Blade Runner (1982)\n8. Matrix, The (1999)\n9. Twelve Monkeys (1995)\n10. Terminator 2: Judgment Day (1991)",F,18-24,college/grad student,1527,"Fifth Element, The (1997)"
387786,"User profile (audit only):\n- gender: M\n- age: 45-49\n- occupation: customer service\n\nWatch history:\n1. Heartbreak Ridge (1986)\n2. Swing Kids (1993)\n3. Force 10 from Navarone (1978)\n4. Small Soldiers (1998)\n5. Transformers: The Movie, The (1986)\n6. Rambo: First Blood Part II (1985)\n7. Red Dawn (1984)\n8. Hot Shots! Part Deux (1993)\n9. Mars Attac

In [12]:
print(len(test_df))
test_df.head()

281943


,prompt,context,gender,age,occupation,target_mid,target_title
568947,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: executive/managerial\n\nWatch history:\n1. Waterboy, The (1998)\n2. Billy Madison (1995)\n3. Vampire in Brooklyn (1995)\n4. Major Payne (1994)\n5. Deuce Bigalow: Male Gigolo (1999)\n6. Cabin Boy (1994)\n7. Black Sheep (1996)\n8. Jerky Boys, The (1994)\n9. Christmas Story, A (1983)\n10. Godfather, The (1972)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Waterboy, The (1998)\n2. Billy Madison (1995)\n3. Vampire in Brooklyn (1995)\n4. Major Payne (1994)\n5. Deuce Bigalow: Male Gigolo (1999)\n6. Cabin Boy (1994)\n7. Black Sheep (1996)\n8. Jerky Boys, The (1994)\n9. Christmas Story, A (1983)\n10. Godfather, The (1972)",M,25-34,executive/managerial,260,Star Wars: Episode IV - A New Hope (1977)
324025,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: executive/managerial\n\nWatch history:\n1. Fast, Cheap & Out of Control (1997)\n2. On the Waterfront (1954)\n3. American History X (1998)\n4. Iron Giant, The (1999)\n5. Who's Afraid of Virginia Woolf? (1966)\n6. Toy Story 2 (1999)\n7. Monty Python's Life of Brian (1979)\n8. Player, The (1992)\n9. Waiting for Guffman (1996)\n10. There's Something About Mary (1998)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Fast, Cheap & Out of Control (1997)\n2. On the Waterfront (1954)\n3. American History X (1998)\n4. Iron Giant, The (1999)\n5. Who's Afraid of Virginia Woolf? (1966)\n6. Toy Story 2 (1999)\n7. Monty Python's Life of Brian (1979)\n8. Player, The (1992)\n9. Waiting for Guffman (1996)\n10. There's Something About Mary (1998)",M,25-34,executive/managerial,2108,L.A. Story (1991)
65694,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: artist\n\nWatch history:\n1. Peacemaker, The (1997)\n2. Red Dawn (1984)\n3. Mars Attacks! (1996)\n4. G.I. Jane (1997)\n5. Transformers: The Movie, The (1986)\n6. McHale's Navy (1997)\n7. Rambo: First Blood Part II (1985)\n8. Iron Eagle (1986)\n9. Hot Shots! Part Deux (1993)\n10. Canadian Bacon (1994)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Peacemaker, The (1997)\n2. Red Dawn (1984)\n3. Mars Attacks! (1996)\n4. G.I. Jane (1997)\n5. Transformers: The Movie, The (1986)\n6. McHale's Navy (1997)\n7. Rambo: First Blood Part II (1985)\n8. Iron Eagle (1986)\n9. Hot Shots! Part Deux (1993)\n10. Canadian Bacon (1994)",F,25-34,artist,3768,Braddock: Missing in Action III (1988)
865015,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: clerical/admin\n\nWatch history:\n1. Moonstruck (1987)\n2. Presidio, The (1988)\n3. Postman Always Rings Twice, The (1981)\n4. In the Heat of the Night (1967)\n5. Mighty Aphrodite (1995)\n6. Erin Brockovich (2000)\n7. 28 Days (2000)\n8. Keeping the Faith (2000)\n9. Gladiator (2000)\n10. Mission to Mars (2000)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Moonstruck (1987)\n2. Presidio, The (1988)\n3. Postman Always Rings Twice, The (1981)\n4. In the Heat of the Night (1967)\n5. Mighty Aphrodite (1995)\n6. Erin Brockovich (2000)\n7. 28 Days (2000)\n8. Keeping the Faith (2000)\n9. Gladiator (2000)\n10. Mission to Mars (2000)",F,25-34,clerical/admin,3565,Where the Heart Is (2000)
533536,"User profile (audit only):\n- gender: M\n- age: 18-24\n- occupation: college/grad student\n\nWatch history:\n1. Topsy-Turvy (1999)\n2. Ronin (1998)\n3. End of the Affair, The (1999)\n4. Fugitive, The (1993)\n5. Jerry Maguire (1996)\n6. Four Weddings and a Funeral (1994)\n7. Star Wars: Episode V

In [13]:
# Build catalog mapper
mapper = CatalogMapper(embedder, loader.item_db)
mapper.build(dedup=True)

2026-01-16 20:44:12,178 - INFO - Building catalog embeddings for 3883 items...
Batches: 100%|██████████| 16/16 [00:06<00:00,  2.56it/s]


In [14]:
train_data_mini = train_df[:3].copy()
train_data_mini

,prompt,context,gender,age,occupation,target_mid,target_title
160383,"User profile (audit only):\n- gender: F\n- age: Under 18\n- occupation: K-12 student\n\nWatch history:\n1. Snow White and the Seven Dwarfs (1937)\n2. Fantasia 2000 (1999)\n3. Cell, The (2000)\n4. Road to El Dorado, The (2000)\n5. Big Momma's House (2000)\n6. Gossip (2000)\n7. Chasing Amy (1997)\n8. Dogma (1999)\n9. What Lies Beneath (2000)\n10. Scream (1996)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Snow White and the Seven Dwarfs (1937)\n2. Fantasia 2000 (1999)\n3. Cell, The (2000)\n4. Road to El Dorado, The (2000)\n5. Big Momma's House (2000)\n6. Gossip (2000)\n7. Chasing Amy (1997)\n8. Dogma (1999)\n9. What Lies Beneath (2000)\n10. Scream (1996)",F,Under 18,K-12 student,920,Gone with the Wind (1939)
181219,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: programmer\n\nWatch history:\n1. Thin Red Line, The (1998)\n2. G.I. Jane (1997)\n3. In the Army Now (1994)\n4. Terminator 2: Judgment Day (1991)\n5. Hunt for Red October, The (1990)\n6. Fugitive, The (1993)\n7. Matrix, The (1999)\n8. Speed (1994)\n9. Jurassic Park (1993)\n10. Total Recall (1990)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Thin Red Line, The (1998)\n2. G.I. Jane (1997)\n3. In the Army Now (1994)\n4. Terminator 2: Judgment Day (1991)\n5. Hunt for Red October, The (1990)\n6. Fugitive, The (1993)\n7. Matrix, The (1999)\n8. Speed (1994)\n9. Jurassic Park (1993)\n10. Total Recall (1990)",M,25-34,programmer,733,"Rock, The (1996)"
35914,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: academic/educator\n\nWatch history:\n1. Honey, I Shrunk the Kids (1989)\n2. Armageddon (1998)\n3. Highlander III: The Sorcerer (1994)\n4. Transformers: The Movie, The (1986)\n5. Conquest of the Planet of the Apes (1972)\n6. Beneath the Planet of the Apes (1970)\n7. Coneheads (1993)\n8. Johnny Mnemonic (1995)\n9. Tank Girl (1995)\n10. Honey, I Blew Up the Kid (1992)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Honey, I Shrunk the Kids (1989)\n2. Armageddon (1998)\n3. Highlander III: The Sorcerer (1994)\n4. Transformers: The Movie, The (1986)\n5. Conquest of the Planet of the Apes (1972)\n6. Beneath the Planet of the Apes (1970)\n7. Coneheads (1993)\n8. Johnny Mnemonic (1995)\n9. Tank Girl (1995)\n10. Honey, I Blew Up the Kid (1992)",F,25-34,academic/educator,2091,Return from Witch Mountain (1978)


In [15]:
test_data_mini = test_df[:3].copy()
test_data_mini

,prompt,context,gender,age,occupation,target_mid,target_title
568947,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: executive/managerial\n\nWatch history:\n1. Waterboy, The (1998)\n2. Billy Madison (1995)\n3. Vampire in Brooklyn (1995)\n4. Major Payne (1994)\n5. Deuce Bigalow: Male Gigolo (1999)\n6. Cabin Boy (1994)\n7. Black Sheep (1996)\n8. Jerky Boys, The (1994)\n9. Christmas Story, A (1983)\n10. Godfather, The (1972)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Waterboy, The (1998)\n2. Billy Madison (1995)\n3. Vampire in Brooklyn (1995)\n4. Major Payne (1994)\n5. Deuce Bigalow: Male Gigolo (1999)\n6. Cabin Boy (1994)\n7. Black Sheep (1996)\n8. Jerky Boys, The (1994)\n9. Christmas Story, A (1983)\n10. Godfather, The (1972)",M,25-34,executive/managerial,260,Star Wars: Episode IV - A New Hope (1977)
324025,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: executive/managerial\n\nWatch history:\n1. Fast, Cheap & Out of Control (1997)\n2. On the Waterfront (1954)\n3. American History X (1998)\n4. Iron Giant, The (1999)\n5. Who's Afraid of Virginia Woolf? (1966)\n6. Toy Story 2 (1999)\n7. Monty Python's Life of Brian (1979)\n8. Player, The (1992)\n9. Waiting for Guffman (1996)\n10. There's Something About Mary (1998)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Fast, Cheap & Out of Control (1997)\n2. On the Waterfront (1954)\n3. American History X (1998)\n4. Iron Giant, The (1999)\n5. Who's Afraid of Virginia Woolf? (1966)\n6. Toy Story 2 (1999)\n7. Monty Python's Life of Brian (1979)\n8. Player, The (1992)\n9. Waiting for Guffman (1996)\n10. There's Something About Mary (1998)",M,25-34,executive/managerial,2108,L.A. Story (1991)
65694,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: artist\n\nWatch history:\n1. Peacemaker, The (1997)\n2. Red Dawn (1984)\n3. Mars Attacks! (1996)\n4. G.I. Jane (1997)\n5. Transformers: The Movie, The (1986)\n6. McHale's Navy (1997)\n7. Rambo: First Blood Part II (1985)\n8. Iron Eagle (1986)\n9. Hot Shots! Part Deux (1993)\n10. Canadian Bacon (1994)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Peacemaker, The (1997)\n2. Red Dawn (1984)\n3. Mars Attacks! (1996)\n4. G.I. Jane (1997)\n5. Transformers: The Movie, The (1986)\n6. McHale's Navy (1997)\n7. Rambo: First Blood Part II (1985)\n8. Iron Eagle (1986)\n9. Hot Shots! Part Deux (1993)\n10. Canadian Bacon (1994)",F,25-34,artist,3768,Braddock: Missing in Action III (1988)


In [16]:
# -------------------------
# Offline calibration (FASTER: use rank-1 from open-ended)
# -------------------------
logger.info("Calibration generation (open-ended Top-K)...")
cal_recs = generate_recommendations(train_data_mini["prompt"].tolist(), system_msg="", tokenizer=tokenizer, model=model)

cal_groups = [
_group_key({k: str(row[k]) for k in Config.PROTECTED_ATTRIBUTES})
for _, row in train_data_mini.iterrows()
]

validator = ConformalFairnessValidator(embedder, item_db=loader.item_db)
validator.calibrate(
cal_contexts=train_data_mini["context"].tolist(),
cal_prompts=train_data_mini["prompt"].tolist(),
cal_groups=cal_groups,
cal_recs=cal_recs,
cal_targets=train_data_mini["target_title"].tolist(),
)

prompt_engine = FairPromptEngine(validator)

# Helper for CFR generation (neutral)
def generate_fn(prompts, system_msg):
    return generate_recommendations(prompts, system_msg, tokenizer, model)

2026-01-16 20:44:18,795 - INFO - Calibration generation (open-ended Top-K)...
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
2026-01-16 20:45:05,697 - INFO - Embedding calibration contexts...
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.89s/it]
2026-01-16 20:45:07,601 - INFO - Embedding calibration rank-1 recommendations...
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.24it/s]
2026-01-16 20:45:07,917 - INFO - Computing calibration S scores...
2026-01-16 20:45:11,469 - INFO - Calibration complete: Q_alpha=1.0302 (n=3)


In [17]:
# -------------------------
# Zero-shot baseline (task-matched open-ended)
# -------------------------
zs_raw = run_zero_shot_openended(test_data_mini, tokenizer, model) # mini for debugging
zs_map = []
zs_valid = []
for recs in zs_raw:
    mr = mapper.map_list(recs, k=Config.TOP_K_RECS, min_sim=0.65)
    zs_map.append(mr.mapped_titles)
    zs_valid.append(mr.valid_at_k)

zs_acc = evaluate_at_k_from_lists(zs_map, test_data_mini["target_title"].tolist(), k=Config.TOP_K_RECS)
zs_validm = evaluate_valid_at_k(zs_valid, k=Config.TOP_K_RECS)
zs_sns = compute_snsr_snsv(test_data_mini.assign(mapped_recs=zs_map), embedder, recs_col="mapped_recs", group_mode="tuple")
zs_cfr = compute_cfr(
    test_data_mini,
    embedder,
    generate_fn=generate_fn,
    system_msg_neutral=NEUTRAL_SYSTEM_PROMPT,
    k=Config.TOP_K_RECS,
    n_samples=min(200, len(test_data_mini)),
    flip_mode="tuple",
    prompt_col="prompt",
)

baseline_block = {
    "ZeroShot_OpenEnded": {
        **zs_acc,
        **zs_validm,
        "SNSR": zs_sns.SNSR,
        "SNSV": zs_sns.SNSV,
        "CFR": zs_cfr.CFR,
        "CFR_valid_rate": zs_cfr.valid_rate,
        "CFR_n_pairs": zs_cfr.n_pairs,
    }
}

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [23]:
# -------------------------
# FACTER iterations
# -------------------------
history = []
for it in range(3):
    prompt_engine.set_iteration(it)

    facter_raw = []
    facter_mapped = []
    facter_valid = []
    is_viol = []
    scores = []
    thresholds = []

    for _, row in test_data_mini.iterrows(): # use only the 3 samples
        attrs = {k: str(row[k]) for k in Config.PROTECTED_ATTRIBUTES}
        g = _group_key(attrs)

        system_msg = prompt_engine.generate_system_prompt(current_group=g)
        user_prompt = prompt_engine.update_prompt(row["prompt"], current_group=g)

        recs = generate_recommendations([user_prompt], system_msg, tokenizer, model)[0]
        # map
        mr = mapper.map_list(recs, k=Config.TOP_K_RECS, min_sim=0.35) # changed min_sim from 0.65 to 0.45 because there were no selected recommendation 
        mapped = mr.mapped_titles

        v, s, q = validator.validate(
            context=row["context"],
            prompt=row["prompt"],
            attrs=attrs,
            recs=mapped,             # IMPORTANT: run validator on mapped titles
            y_true_title=row["target_title"],
        )

        facter_raw.append(recs)
        facter_mapped.append(mapped)
        facter_valid.append(mr.valid_at_k)
        is_viol.append(v)
        scores.append(s)
        thresholds.append(q)

    eval_df = test_data_mini.copy()

    eval_df["mapped_recs"] = facter_mapped
    eval_df["valid_at_k"] = facter_valid
    eval_df["is_violation"] = is_viol
    eval_df["S"] = scores
    eval_df["Q"] = thresholds

    viol_rate = float(np.mean(is_viol)) if is_viol else 0.0
    acc = evaluate_at_k_from_lists(facter_mapped, eval_df["target_title"].tolist(), k=Config.TOP_K_RECS)
    validm = evaluate_valid_at_k(facter_valid, k=Config.TOP_K_RECS)

    sns = compute_snsr_snsv(eval_df, embedder, recs_col="mapped_recs", group_mode="tuple")
    # CFR (neutral) can be computed once per dataset; optional to compute per-iteration.
    # Here we compute once in iteration 0 for speed; set to None otherwise.
    cfr = None
    if it == 0:
        cfr = compute_cfr(
            eval_df,
            embedder,
            generate_fn=generate_fn,
            system_msg_neutral=NEUTRAL_SYSTEM_PROMPT,
            k=Config.TOP_K_RECS,
            n_samples=min(200, len(eval_df)),
            flip_mode="tuple",
            prompt_col="prompt",
        )

    record = {
        "iteration": it + 1,
        "violation_rate": viol_rate,
        **acc,
        **validm,
        "SNSR": sns.SNSR,
        "SNSV": sns.SNSV,
        "Q_last": float(eval_df["Q"].iloc[-1]),
    }
    if cfr is not None:
        record.update({"CFR": cfr.CFR, "CFR_valid_rate": cfr.valid_rate, "CFR_n_pairs": cfr.n_pairs})

    logger.info(f"Iter {it+1}: {json.dumps(record, indent=2)}")
    history.append(record)

    if it >= 2 and viol_rate < 0.10:
        break

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attentio

In [24]:
eval_df

,prompt,context,gender,age,occupation,target_mid,target_title,mapped_recs,valid_at_k,is_violation,S,Q
568947,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: executive/managerial\n\nWatch history:\n1. Waterboy, The (1998)\n2. Billy Madison (1995)\n3. Vampire in Brooklyn (1995)\n4. Major Payne (1994)\n5. Deuce Bigalow: Male Gigolo (1999)\n6. Cabin Boy (1994)\n7. Black Sheep (1996)\n8. Jerky Boys, The (1994)\n9. Christmas Story, A (1983)\n10. Godfather, The (1972)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Waterboy, The (1998)\n2. Billy Madison (1995)\n3. Vampire in Brooklyn (1995)\n4. Major Payne (1994)\n5. Deuce Bigalow: Male Gigolo (1999)\n6. Cabin Boy (1994)\n7. Black Sheep (1996)\n8. Jerky Boys, The (1994)\n9. Christmas Story, A (1983)\n10. Godfather, The (1972)",M,25-34,executive/managerial,260,Star Wars: Episode IV - A New Hope (1977),"[, Rules of Engagement (2000), , , , All Things Fair (1996), , , Gendernauts (1999), ]",0.3,False,0.932708,1.227345
324025,"User profile (audit only):\n- gender: M\n- age: 25-34\n- occupation: executive/managerial\n\nWatch history:\n1. Fast, Cheap & Out of Control (1997)\n2. On the Waterfront (1954)\n3. American History X (1998)\n4. Iron Giant, The (1999)\n5. Who's Afraid of Virginia Woolf? (1966)\n6. Toy Story 2 (1999)\n7. Monty Python's Life of Brian (1979)\n8. Player, The (1992)\n9. Waiting for Guffman (1996)\n10. There's Something About Mary (1998)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Fast, Cheap & Out of Control (1997)\n2. On the Waterfront (1954)\n3. American History X (1998)\n4. Iron Giant, The (1999)\n5. Who's Afraid of Virginia Woolf? (1966)\n6. Toy Story 2 (1999)\n7. Monty Python's Life of Brian (1979)\n8. Player, The (1992)\n9. Waiting for Guffman (1996)\n10. There's Something About Mary (1998)",M,25-34,executive/managerial,2108,L.A. Story (1991),"[, Rules of Engagement (2000), , , , All Things Fair (1996), , , Gendernauts (1999), ]",0.3,False,0.874277,1.227345
65694,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: artist\n\nWatch history:\n1. Peacemaker, The (1997)\n2. Red Dawn (1984)\n3. Mars Attacks! (1996)\n4. G.I. Jane (1997)\n5. Transformers: The Movie, The (1986)\n6. McHale's Navy (1997)\n7. Rambo: First Blood Part II (1985)\n8. Iron Eagle (1986)\n9. Hot Shots! Part Deux (1993)\n10. Canadian Bacon (1994)\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Peacemaker, The (1997)\n2. Red Dawn (1984)\n3. Mars Attacks! (1996)\n4. G.I. Jane (1997)\n5. Transformers: The Movie, The (1986)\n6. McHale's Navy (1997)\n7. Rambo: First Blood Part II (1985)\n8. Iron Eagle (1986)\n9. Hot Shots! Part Deux (1993)\n10. Canadian Bacon (1994)",F,25-34,artist,3768,Braddock: Missing in Action III (1988),"[, Rules of Engagement (2000), , , , All Things Fair (1996), , , Male and Female (1919), ]",0.3,True,1.608474,1.257835


In [25]:
record

{'iteration': 3,
 'violation_rate': 0.3333333333333333,
 'HitRate@10': 0.0,
 'NDCG@10': 0.0,
 'Valid@10': 0.3,
 'SNSR': 0.0,
 'SNSV': 0.0,
 'Q_last': 1.2578354460221512}